# Hugging Face Applications — Lesson 4: Translation

> Learning material for **Hugging Face Applications**. Companion to the lesson script `04_Translation.py` (same content, runnable without Jupyter).

**Task ID:** HF-204  |  **Folder:** `documentation`


## What is machine translation?

A **translation model** converts text from one language into another, keeping the meaning:

> 🇬🇧 *"The weather is nice today."*  →  🇫🇷 *"Le temps est beau aujourd'hui."*

## One model per language pair

The classic approach (OPUS-MT by Helsinki-NLP) trains **a separate model for every pair**: `opus-mt-en-fr` translates English→French *only*, `opus-mt-en-de` English→German, and so on. You pick the model that matches your pair (hundreds exist on the Hub).

> **Analogy:** a human translator usually has one pair of languages they can do well.

## The model architecture: seq2seq again

Translation models use the same encoder→decoder design as summarization (HF-202): encoder reads the source sentence, decoder writes the target sentence word by word.

## transformers v5 note

The `pipeline("translation_en_to_fr")` shortcut was removed in v5 — the **tokenize → generate → decode** loop here is the supported way.

**Step 1 — load the en→fr model** (about 300 MB):


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("Loaded", model_name)


**Step 2 — translate:**

In [ ]:
text = "Hugging Face builds open-source tools for artificial intelligence."

inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
output_ids = model.generate(**inputs, max_new_tokens=512)
translation = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Source:    ", text)
print("Translated:", translation)


## Sentences vs. paragraphs: batching

The tokenizer can take a *list* of sentences; the model translates them together. Long paragraphs are fine too — but one huge text in a single call can exceed the model limit (`max_length=512` clips it).


In [ ]:
sentences = [
    "Good morning, how are you?",
    "Machine learning is fun.",
    "See you tomorrow!",
]
inputs = tokenizer(sentences, return_tensors="pt", truncation=True, padding=True)
outs = model.generate(**inputs, max_new_tokens=64)
for s, o in zip(sentences, outs):
    print(f"{s}  ->  {tokenizer.decode(o, skip_special_tokens=True)}")


## Try it yourself

1. Change direction: load `Helsinki-NLP/opus-mt-fr-en` and translate *"Le temps est beau."*
2. Try `Helsinki-NLP/opus-mt-en-hi` with an English greeting.
3. Translate a short paragraph — then translate it *back* with the reverse model. Is it identical? (Round-trip is a classic quality test.)

## Common pitfalls

- **Using the wrong direction model** (`en-fr` for French *to* English) — output will be garbage.
- **Download once, not per sentence** — load the model once in your script.
- Some pairs need `sacremoses` (pip package) for their tokenizer — if the tokenizer warns, install it.

## Summary

- Translation is seq2seq: encoder reads the source, decoder writes the target.
- OPUS-MT = one model per pair; pick the right id on the Hub.

**Next lesson:** HF-205 — Text Classification.  |  Extra reading: `../resources/reference_links.md`
